In [3]:
!pip install transformers torch pandas -q

In [12]:
import pandas as pd
import torch
import re
import csv
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

ANSI_RESET = "\033[0m"
ANSI_BOLD = "\033[1m"
ANSI_CYAN = "\033[96m"
ANSI_GREEN = "\033[92m"

print("=" * 70)
print(f"{ANSI_BOLD}Loading Text-to-Text Model (facebook/bart-large-cnn){ANSI_RESET}")
print("=" * 70)

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# High quality news summarization Seq2Seq Model
MODEL_NAME = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

df = pd.read_csv("1500news.csv")

# Strict Text Cleaning to fix CSV data mixing
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'[\r\n\t"\']+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Clean input columns
df['title'] = df['title'].apply(clean_text)
df['first_paragraph'] = df['first_paragraph'].apply(clean_text)

titles = df['title'].tolist()
paragraphs = df['first_paragraph'].tolist()

# Combine Title & First Paragraph for rich context
inputs = [f"{t}. {p}" for t, p in zip(titles, paragraphs)]

print("\nGenerating High Quality Summaries...")

generated_summaries = []
batch_size = 16  # Optimal batch size for GPU

for i in range(0, len(inputs), batch_size):
    batch_text = inputs[i:i + batch_size]

    # Tokenize input
    tokenized = tokenizer(
        batch_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    # Abstractive Text Generation (Seq2Seq / Decoder Beam Search)
    summary_ids = model.generate(
        tokenized["input_ids"],
        num_beams=4,
        max_length=60,
        min_length=15,
        length_penalty=2.0,
        early_stopping=True
    )

    decoded = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
    generated_summaries.extend(decoded)

# Clean summaries
df['summary'] = [clean_text(s) for s in generated_summaries]

print("=" * 70)
print(f"{ANSI_BOLD}Previewing Clean Generated Summaries (First 5 Rows){ANSI_RESET}")
print("=" * 70)

for idx in range(5):
    print(f"{ANSI_CYAN}Title:{ANSI_RESET} {df['title'].iloc[idx]}")
    print(f"📝 {ANSI_GREEN}Summary:{ANSI_RESET} {df['summary'].iloc[idx]}")
    print("-" * 70)

output_filename = "1500news_high_quality_summaries.csv"

# Safe CSV Export (Prevents Data Mixing in Excel)
df.to_csv(
    output_filename,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_MINIMAL
)

print(f"\n{ANSI_GREEN}File saved successfully as: {output_filename}{ANSI_RESET}")

# Download in Colab
from google.colab import files
files.download(output_filename)

Loading Text-to-Text Model (facebook/bart-large-cnn)


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

...

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]


Generating High Quality Summaries...
Previewing Clean Generated Summaries (First 5 Rows)
Title: How to run for reelection in a messy Congress? It’s not easy
📝 Summary: President. Barack Obama is the president of the United States. He is also the leader of the U.S. House of Representatives.
----------------------------------------------------------------------
Title: Ishan Kishan sets up 90-run win over Zimbabwe as India clinches T20 series
📝 Summary: Ishan Kishan sets up a T20 series with a game to spare on Saturday. The Zimbabwean is the nation s most successful T20 team.
----------------------------------------------------------------------
Title: Tate brothers will remain in jail at least 2 more weeks as they fight extradition to the UK
📝 Summary: Andrew and Tristan are accused of trying to get to the United Kingdom to help them with a crime. They are also accused of helping them get to a federal judge to get them to the UK.
---------------------------------------------------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>